## Problem: Bridging Trade-Level Data to Day-Level Sentiment

The trader dataset (`historical_data.csv`) contains 211,224 rows, where each row 
represents a single trade execution — with fields like Account, Execution Price, 
Size USD, and Closed PnL.

The sentiment dataset (`fear_greed_index.csv`) contains 2,644 rows, where each row 
represents one sentiment label (Fear/Greed) for a single calendar date.

These two datasets exist at different levels of granularity: the trader data is 
per-trade (many rows per account per day), while the sentiment data is per-day 
(exactly one row per date, across all traders).

To compare trader behavior against market sentiment, both datasets must exist at 
the same level of granularity — per account, per day. This requires collapsing 
the trade-level data into a summary table, where each row represents one account's 
aggregated trading activity for one specific day.



In [16]:
import pandas as pd
import numpy as np
from datetime import datetime

In [3]:
df=pd.read_csv("/home/harsh/Desktop/python/primetrade_assignment/historical_data.csv")
df_sentiment=pd.read_csv("/home/harsh/Desktop/python/primetrade_assignment/fear_greed_index.csv")


In [4]:
df

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0000,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211219,0x72743ae2822edd658c0c50608fd7c5c501b2afbd,FARTCOIN,1.1010,382.20,420.80,SELL,25-04-2025 15:35,7546.600000,Close Long,-20.2566,0xcd339c08dc7b615a993c0422374d8e02027400092bc2...,88803313862,False,0.042080,1.990000e+14,1.750000e+12
211220,0x72743ae2822edd658c0c50608fd7c5c501b2afbd,FARTCOIN,1.1010,2124.10,2338.63,SELL,25-04-2025 15:35,7164.400000,Close Long,-112.5773,0x29e8ede2a3a37aa0eac00422374d8e02029b00ac9f3c...,88803313862,False,0.233863,9.260000e+14,1.750000e+12
211221,0x72743ae2822edd658c0c50608fd7c5c501b2afbd,FARTCOIN,1.1010,423.40,466.16,SELL,25-04-2025 15:35,5040.300000,Close Long,-22.4402,0x0780085b0c0a943eea800422374d920204c100edf579...,88803313862,False,0.046616,6.930000e+14,1.750000e+12
211222,0x72743ae2822edd658c0c50608fd7c5c501b2afbd,FARTCOIN,1.1010,3599.80,3963.38,SELL,25-04-2025 15:35,4616.900000,Close Long,-190.7894,0x349c29934913b25c89e20422374d920204cd008b8a0e...,88803313862,False,0.396337,4.180000e+14,1.750000e+12


In [11]:
df['Timestamp IST'] = pd.to_datetime(df['Timestamp IST'], format='%d-%m-%Y %H:%M')
df['Timestamp IST']

0        2024-12-02 22:50:00
1        2024-12-02 22:50:00
2        2024-12-02 22:50:00
3        2024-12-02 22:50:00
4        2024-12-02 22:50:00
                 ...        
211219   2025-04-25 15:35:00
211220   2025-04-25 15:35:00
211221   2025-04-25 15:35:00
211222   2025-04-25 15:35:00
211223   2025-04-25 15:35:00
Name: Timestamp IST, Length: 211224, dtype: datetime64[us]

In [13]:
df.describe()

,Execution Price,Size Tokens,Size USD,Timestamp IST,Start Position,Closed PnL,Order ID,Fee,Trade ID,Timestamp
count,211224.000000,2.112240e+05,2.112240e+05,211224,2.112240e+05,211224.000000,2.112240e+05,211224.000000,2.112240e+05,2.112240e+05
mean,11414.723350,4.623365e+03,5.639451e+03,2025-01-31 12:04:22.915010,-2.994625e+04,48.749001,6.965388e+10,1.163967,5.628549e+14,1.737744e+12
min,0.000005,8.740000e-07,0.000000e+00,2023-05-01 01:06:00,-1.433463e+07,-117990.104100,1.732711e+08,-1.175712,0.000000e+00,1.680000e+12
25%,4.854700,2.940000e+00,1.937900e+02,2024-12-31 21:00:45,-3.762311e+02,0.000000,5.983853e+10,0.016121,2.810000e+14,1.740000e+12
50%,18.280000,3.200000e+01,5.970450e+02,2025-02-24 18:55:00,8.472793e+01,0.000000,7.442939e+10,0.089578,5.620000e+14,1.740000e+12
75%,101.580000,1.879025e+02,2.058960e+03,2025-04-02 18:22:00,9.337278e+03,5.792797,8.335543e+10,0.393811,8.460000e+14,1.740000e+12
max,109004.000000,1.582244e+07,3.921431e+06,2025-05-01 12:13:00,3.050948e+07,135329.090100,9.014923e+10,837.471593,1.130000e+15,1.750000e+12
std,29447.654868,1.042729e+05,3.657514e+04,NaN,6.738074e+05,919.164828,1.835753e+10,6.758854,3.257565e+14,8.689920e+09


In [25]:
df.shape

(211224, 16)

In [14]:
df.isnull().sum()

Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

In [26]:
df.duplicated().sum()

np.int64(0)

In [35]:
df['date'] = df['Timestamp IST'].dt.date
df['date']=pd.to_datetime(df['date'])
df['date']

0        2024-12-02
1        2024-12-02
2        2024-12-02
3        2024-12-02
4        2024-12-02
            ...    
211219   2025-04-25
211220   2025-04-25
211221   2025-04-25
211222   2025-04-25
211223   2025-04-25
Name: date, Length: 211224, dtype: datetime64[s]

In [17]:
df_sentiment

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05
...,...,...,...,...
2639,1745818200,54,Neutral,2025-04-28
2640,1745904600,60,Greed,2025-04-29
2641,1745991000,56,Greed,2025-04-30
2642,1746077400,53,Neutral,2025-05-01


In [21]:
df_sentiment['date']=pd.to_datetime(df_sentiment['date'])
df_sentiment['date']

0      2018-02-01
1      2018-02-02
2      2018-02-03
3      2018-02-04
4      2018-02-05
          ...    
2639   2025-04-28
2640   2025-04-29
2641   2025-04-30
2642   2025-05-01
2643   2025-05-02
Name: date, Length: 2644, dtype: datetime64[us]

In [27]:
df_sentiment.shape

(2644, 4)

In [22]:
df_sentiment.isnull().sum()

timestamp         0
value             0
classification    0
date              0
dtype: int64

In [30]:
df_sentiment.duplicated().sum()

np.int64(0)

In [23]:
df_sentiment.describe()

,timestamp,value,date
count,2.644000e+03,2644.000000,2644
mean,1.631899e+09,46.981089,2021-09-17 11:44:45.022693
min,1.517463e+09,5.000000,2018-02-01 00:00:00
25%,1.574811e+09,28.000000,2019-11-26 18:00:00
50%,1.631900e+09,46.000000,2021-09-17 12:00:00
75%,1.688989e+09,66.000000,2023-07-10 06:00:00
max,1.746164e+09,95.000000,2025-05-02 00:00:00
std,6.597967e+07,21.827680,NaN


In [ ]:
df['is_win'] = df['Closed PnL'] > 0
df['is_long'] = df['Side'] == 'BUY'
daily_metrics = df.groupby(['Account', 'date']).agg( 
                                                    daily_pnl = ('Closed PnL', 'sum'),
                                                    avg_trade_size = ('Size USD', 'mean'), 
                                                    trades_per_day = ('Closed PnL', 'count'),
                                                    win_rate = ('is_win', 'mean'),
                                                    long_ratio = ('is_long', 'mean') ).sort_values('date').reset_index()

### Why this aggregation

Sentiment is known once per day, but an account can trade many times in a day — 
so trade data needs to be collapsed to one row per account per day before it can 
be compared to sentiment.

Grouping by `Account` and `date` buckets all trades from the same account on the 
same day. Within each bucket: `daily_pnl` sums `Closed PnL`  (net result for the day), 
`avg_trade_size` averages Size USD (typical position size), and `trades_per_day` 
counts trades (activity level). This gives one row per account-day, ready to merge 
with sentiment on `date`.

In [73]:
daily_metrics.shape

(2341, 7)

In [74]:
daily_metrics.describe()

,date,daily_pnl,avg_trade_size,trades_per_day,win_rate,long_ratio
count,2341,2341.000000,2341.000000,2341.000000,2341.000000,2341.000000
mean,2024-12-22 00:49:49,4398.530091,6989.515321,90.228108,0.359926,0.489142
min,2023-05-01 00:00:00,-358963.139984,0.000000,1.000000,0.000000,0.000000
25%,2024-11-28 00:00:00,0.000000,695.250952,9.000000,0.000000,0.142857
50%,2025-01-28 00:00:00,207.983482,1914.000000,29.000000,0.318182,0.486486
75%,2025-03-18 00:00:00,1842.839943,7051.005833,80.000000,0.608000,0.833333
max,2025-05-01 00:00:00,533974.662903,844654.190000,4083.000000,1.000000,1.000000
std,NaN,28415.938999,21538.691665,214.611751,0.343601,0.364306


### Sanity check on daily_metrics

After aggregating trades to account-day level, `daily_metrics` contains 2,341 
rows across 2023-05-01 to 2025-05-01.

- **daily_pnl** ranges from -358,963 to 533,974, with a mean of ~4,399. The min 
  is larger in magnitude than any single trade's PnL (-117,990 from the raw data), 
  which makes sense — a day's total is the sum of multiple trades, so it can 
  exceed any individual trade's loss or gain.
- **avg_trade_size** ranges from 0 to 844,654, with a median of 1,914 — most 
  accounts trade modestly, but a few place very large positions.
- **trades_per_day** ranges from 1 to 4,083, with a median of 29. The max of 
  4,083 trades in a single day likely reflects a high-frequency or bot-driven 
  account, checked and retained as a legitimate outlier, not a data error.

No nulls or obviously invalid values were found. The wide spread across all 
three metrics reflects real diversity in trader behavior — from small, 
infrequent traders to large-volume, high-frequency accounts — which is expected 
in this dataset and preserved for the segmentation analysis in Part B.

You can't directly aggregate Closed PnL into a win rate with .agg() the way you did for sum/mean/count, because win rate isn't a property of the raw PnL numbers — it's a property of how many trades were wins, which requires a decision first: what counts as a "win"?


`is_long = Side == 'BUY'` marks each trade as long or short, based on Side. This 
simplifies "Sell" as always short, though in practice it can also mean closing 
a long position — the dataset doesn't distinguish this. Its mean gives the 
proportion of trades that were long, used as the long/short ratio.

In [75]:
merged = daily_metrics.merge(df_sentiment[['date', 'classification']], on='date', how='left')

### Merging with sentiment data

The trader data and sentiment data now share the same granularity (it means level of detail at which each row in your data represents something) — one row per 
account per day. This allows a merge on `date`, attaching each account-day's 
Fear/Greed classification.

A **left join** is used instead of an inner join: it keeps every row from 
`daily_metrics` regardless of whether a sentiment match exists, so any missing 
matches show up as `NaN` and can be checked explicitly, rather than being 
silently dropped. Since the sentiment dataset's date range (2018–2025) fully 
covers the trader dataset's range (2023–2025), missing matches are expected 
to be zero or near-zero — verified below.

In [76]:
merged

,Account,date,daily_pnl,avg_trade_size,trades_per_day,win_rate,long_ratio,classification
0,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,2023-05-01,0.000000,159.000000,3,0.000000,1.000000,Greed
1,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2023-12-05,0.000000,5556.203333,9,0.000000,0.777778,Extreme Greed
2,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2023-12-14,-205.434737,10291.213636,11,0.363636,0.454545,Greed
3,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2023-12-15,-24.632034,5304.975000,2,0.000000,1.000000,Greed
4,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,2023-12-16,0.000000,5116.256667,3,0.000000,1.000000,Greed
...,...,...,...,...,...,...,...,...
2336,0xbaaaf6571ab7d571043ff1e313a9609a10637864,2025-05-01,1.860320,3.900000,1,1.000000,0.000000,Neutral
2337,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,2025-05-01,1243.853569,815.505760,250,0.324000,0.676000,Neutral
2338,0x8477e447846c758f5a675856001ea72298fd9cb5,2025-05-01,-200.948460,309.656842,19,0.000000,0.736842,Neutral
2339,0x4f93fead39b70a1824f981a54d4e55b278e9f760,2025-05-01,1220.746604,14373.198571,14,0.500000,0.214286,Neutral


In [87]:
merged['classification'].isnull().sum()

np.int64(0)

In [84]:
merged.shape

(2341, 8)

In [85]:
merged[merged['classification'].isnull()]

,Account,date,daily_pnl,avg_trade_size,trades_per_day,win_rate,long_ratio,classification
482,0x72c6a4624e1dffa724e6d00d64ceae698af892a0,2024-10-26,42471.99413,14778.143333,6,1.0,1.0,NaN


In [86]:
merged = merged.dropna(subset=['classification'])

### Handling the one unmatched row

One row (out of 2,341) had no matching sentiment classification after the merge. 
Checking its date confirmed it falls at the edge of the trader data's range, 
where a sentiment match was unavailable. Given this affects only 0.04% of rows, 
it was dropped rather than imputed, to avoid introducing an artificial sentiment 
label for a genuine data gap.